# KryxAI risk model — training

Trains the optional ONNX risk model that `kryxai.scoring.fusion.LearnedRiskModel`
loads, and writes the model card that `install_model.py` checks.

**What this notebook does not do.** It does not probe a network, capture real
traffic, or contact any mail server. Every byte it trains on is generated by
`training/dataset.py` inside this repository. There is no real citizen mail in
this model, and the resulting scores say nothing about any real deployment.

**The one rule that makes the numbers meaningful.** The target label comes from
the weakness profile that was *injected* when the capture was generated, not
from what KryxAI's rule engine concludes about it. If the label were "did the
engine raise a high finding", the model would merely learn to imitate
`policy/kb.py`, would score near-perfectly on data derived from those rules, and
that score would mean nothing. The label is what a reviewer would say on looking
at the handshake.

**Requirement.** Fill in `REPO_URL` below, or set it in a cell before running.

## 1. Parameters

In [ ]:
REPO_URL = ""  # e.g. "https://github.com/<org>/kryxai.git"
REPO_BRANCH = "main"
NOTEBOOK_TAG = "2026-09-27"

import os, sys, subprocess
from pathlib import Path

# Accepting the URL from the environment avoids having to edit this cell if the
# repository moves. Nothing secret is involved either way.
REPO_URL = os.environ.get("KRYXAI_REPO_URL") or REPO_URL
if not REPO_URL:
    raise SystemExit(
        "Set REPO_URL in the cell above (or export KRYXAI_REPO_URL) and re-run."
    )

HERE = Path.cwd()
if not (HERE / "kryxai").is_dir():
    subprocess.run(
        ["git", "clone", "--depth", "1", "-b", REPO_BRANCH, REPO_URL, "kryxai-src"],
        check=True,
    )
    HERE = HERE / "kryxai-src"
else:
    HERE = HERE / "kryxai"

sys.path.insert(0, str(HERE))
print("working from", HERE)

## 2. Install

`[all]` pulls the analysis stack, the PDF reporter and the API server.
`onnxruntime` is separate because it is only needed for the optional model.

In [ ]:
!pip install -q -e "{HERE}[all,dev]" onnxruntime onnx scikit-learn

## 3. Confirm the feature contract before training

The model is only loadable if it declares the same schema version and the same
feature names, in the same order, as this build of KryxAI. A mismatch is
refused at install time, so it is worth checking now rather than after a
training run.

Note the log-compressed columns. `cert_key_bits` and the byte/day counts are
unbounded integers, so `features_for` applies `log1p` to them. That transform
lives in the package, not here, precisely so the vector a model is trained on
and the vector the runtime feeds it come from one piece of code.

In [ ]:
from kryxai.scoring import fusion

schema = fusion.feature_schema()
print("schema version :", schema["version"])
print("feature count  :", schema["count"])
for name in schema["names"]:
    print("   ", name)

assert len(schema["names"]) == len(set(schema["names"])), "duplicate feature name"
print("\nlog-compressed inside features_for:", fusion.BOUNDED_FEATURES)

## 4. Generate the labelled corpus

Each capture is built around a declared weakness profile, and the label is that
profile. Groups are certificate identities: the same identity never appears in
both the train and test halves, so a model cannot score well by recognising a
certificate it has already seen.

In [ ]:
import sys, shutil
from collections import Counter
from pathlib import Path

sys.path.insert(0, str(HERE / "training"))
import dataset as D

DATA = HERE / "training" / "data"
if DATA.exists():
    shutil.rmtree(DATA)
DATA.mkdir(parents=True)

samples = D.write_corpus(DATA)
print(f"{len(samples)} captures written to {DATA}\n")

for band in D.BANDS:
    print(f"  {band:12} {sum(1 for s in samples if s.band == band):3}")
print()
groups = Counter(s.group for s in samples)
print("groups (certificate identities):", dict(groups))
assert len(groups) >= 7, "too few groups for a group-aware split to mean anything"

## 5. Build the feature matrix

One row per session, via `fusion.session_feature_row`. No scaling or transform
is applied here: the function already returns the exact vector the runtime will
feed the model, and touching it here is how training and inference drift apart.

A session with no findings still produces a row. The three `ok` captures generate
no findings at all, and a dataset built from findings alone would contain no
clean examples and could not learn what acceptable looks like.

In [ ]:
import numpy as np
from kryxai.config import Settings
from kryxai.engine import run_scan
from kryxai.scoring import fusion

names = fusion.FEATURE_NAMES

X, y, groups, files, profiles = [], [], [], [], []
FINDINGS_BY_ROW = []   # kept for the rule baseline in the next-but-one cell

for s in samples:
    report = run_scan(s.path, Settings(blockchain_difficulty=1), persist=False).report
    assert report["sessions"], f"{s.profile}: no sessions parsed"
    for sess in report["sessions"]:
        fs = [f for f in report["findings"] if f.get("session_id") == sess["session_id"]]
        row = fusion.session_feature_row(sess, fs, {})
        assert len(row) == len(names), f"{s.profile}: wrong feature count"
        assert all(np.isfinite(row)), f"{s.profile}: non-finite feature"
        X.append(row)
        y.append(s.label)
        groups.append(s.group)
        files.append(s.path.name)
        profiles.append(s.profile)
        FINDINGS_BY_ROW.append(fs)

X = np.asarray(X, dtype=np.float32)
y = np.asarray(y)
groups = np.asarray(groups)
print("X", X.shape, "y", y.shape)
print("class counts:", dict(Counter(y.tolist())))
print("range", float(X.min()), "->", float(X.max()))

# If any two bands were not separable from the features, the model would be
# training on noise. Worth seeing before fitting rather than after.
for label, band in enumerate(D.BANDS):
    m = X[y == label].mean(axis=0)
    print(f"  {band:12} n={int((y == label).sum()):3}"
          f"  cert_key_bits={m[names.index('cert_key_bits')]:6.2f}"
          f"  forward_secrecy={m[names.index('tls_forward_secrecy')]:.2f}"
          f"  key_weak={m[names.index('cert_key_weak')]:.2f}"
          f"  expired={m[names.index('cert_expired')]:.2f}"
          f"  starttls_upgraded={m[names.index('starttls_state_upgraded')]:.2f}")

## 6. Split

The split is stratified **and** grouped.

- Grouped, so a certificate identity never appears in both halves. Otherwise a
  model can score well by recognising a certificate it has already seen.
- Stratified, so every band appears in both halves. Grouping alone is not
  enough: with one group per weakness profile, a held-out group can be an
  entire category, and the model is then asked to classify something it has
  never seen. That is not a hard split, it is a broken experiment.

This exact failure occurred in an earlier draft of this notebook, where the
test set contained only the `ok` and one `weak_crypto` band and the model had
never seen a healthy session at all.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

# Four folds; take the first as the test half. Every identity lands in exactly
# one fold, and StratifiedGroupKFold keeps the band proportions even across it.
folds = list(StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=17)
             .split(X, y, groups=groups))
train_idx, test_idx = folds[0]

tr_g, te_g = set(groups[train_idx]), set(groups[test_idx])
assert not (tr_g & te_g), "an identity leaked across the split"
assert set(y[train_idx]) == set(y[test_idx]) == set(range(len(D.BANDS))), \
    "a band is missing from one half of the split"

print(f"train {len(train_idx)} rows / {len(tr_g)} identities")
print(f"test  {len(test_idx)} rows / {len(te_g)} identities")
print("train classes:", dict(Counter(y[train_idx].tolist())))
print("test  classes:", dict(Counter(y[test_idx].tolist())))
print("\nevery band is represented in both halves, and no identity is in both")

## 7. Train

The target is scaled to 0–1 so that a final sigmoid is the correct output and
the exported graph needs no extra output layer. One hidden layer keeps the
graph small enough to read, which matters more than squeezing accuracy out of
synthetic data.

There is exactly one model here. An earlier draft trained one model for scoring
and a second for export, which is how the two drift apart; the exported graph is
now folded from this same pipeline.

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

TARGET_SCALE = len(D.BANDS) - 1

risk_model = Pipeline([
    ("scale", StandardScaler()),
    ("mlp", MLPRegressor(hidden_layer_sizes=(64, 32), activation="relu",
                         alpha=1e-3, max_iter=4000, random_state=17)),
])
risk_model.fit(X[train_idx], (y[train_idx] / TARGET_SCALE).astype("float32"))
print("trained on", len(train_idx), "rows")

## 8. Did it learn, and is it better than reading the rules?

Two checks, and both matter.

The first is held-out accuracy. The second compares against the rule baseline:
the severity the rule engine already assigned, mapped onto the same scale.

**Expect the baseline to be strong, and expect that to be the finding.** The
labels here are injected weakness profiles, and KryxAI's rules were written to
detect exactly those weaknesses. On this corpus the rule engine is a near-perfect
classifier, so a learned model can at best tie with it. A model that cannot
reach parity is not earning its place in the report.

What this experiment *cannot* show is that a model generalises to real mail
traffic, because there is none here. The honest reading of a good result is "the
pipeline is reproducible and the model is no worse than the rules", not "the
model is better than the rules". Establishing that would need labelled real
captures, which this project does not have.

If the model does not reach parity, leave `risk_model.onnx` uninstalled and let
KryxAI run on the rules alone.

In [ ]:
from sklearn.metrics import mean_absolute_error, accuracy_score

def band_pred(idx):
    return np.clip(risk_model.predict(X[idx]) * TARGET_SCALE, 0, TARGET_SCALE)

pred = band_pred(test_idx)
truth = y[test_idx]

def rule_baseline(idx):
    """The rules' own verdict, on the same 0/1/2 scale, for comparison."""
    out = np.zeros(len(idx), dtype=float)
    for j, i in enumerate(idx):
        fs = FINDINGS_BY_ROW[int(i)]
        if not fs:
            out[j] = 0.0
        elif any(f.get("code", "").startswith("starttls") for f in fs):
            out[j] = 2.0
        elif any(f.get("severity") in ("critical", "high") for f in fs):
            out[j] = 1.0
        else:
            out[j] = 1.0
    return out

base = rule_baseline(test_idx)

model_mae = float(mean_absolute_error(truth, pred))
base_mae = float(mean_absolute_error(truth, base))
print("model          MAE", round(model_mae, 4),
      " acc", round(float(accuracy_score(truth, np.round(pred))), 4))
print("rule baseline  MAE", round(base_mae, 4),
      " acc", round(float(accuracy_score(truth, base)), 4))
print()
REACHES_PARITY = model_mae <= base_mae + 0.25
print("reaches parity with the rules (MAE within 0.25 bands):", bool(REACHES_PARITY))
print()
if not REACHES_PARITY:
    print("The model does not match the rules on held-out data. Do not install it:")
    print("KryxAI's validated path is the rule engine alone.")
else:
    print("The model matches the rules on synthetic data. That is a statement about")
    print("this corpus, not about real mail traffic. Installing it is optional and")
    print("the report will show both scores so any disagreement stays visible.")

## 9. Export to ONNX

The StandardScaler is folded into the first layer, so the exported graph takes
**raw** features. This is not an optimisation: `LearnedRiskModel.score()` at
runtime is handed a raw vector from `features_for` and has no scaler of its own,
so a graph expecting pre-scaled input would be fed wrong numbers on every call
and would still produce a plausible number.

The schema version and the feature names go into the model metadata.
`install_model.py` refuses any model whose metadata does not match this build.

In [ ]:
import numpy as np
import onnx
from onnx import helper, TensorProto

scaler = risk_model.named_steps["scale"]
mlp = risk_model.named_steps["mlp"]

mean = scaler.mean_.astype("float64")
sd = np.sqrt(np.where(scaler.var_ > 0, scaler.var_, 1.0)).astype("float64")

# x_scaled = (x - mean) / sd
# x_scaled @ W0 + b0  ==  x @ (W0 / sd[:, None]) + (b0 - mean @ (W0 / sd[:, None]))
# The scale divides the columns of W0, so the offset is mean @ W0_folded.
# Writing it as (mean @ W0) / sd is a shape error: that is a (hidden,) vector
# divided by a (features,) one.
W0 = mlp.coefs_[0].astype("float64")
b0 = mlp.intercepts_[0].astype("float64")
W0_folded = W0 / sd[:, None]
weights = [W0_folded] + [c.astype("float64") for c in mlp.coefs_[1:]]
biases = [b0 - mean @ W0_folded] + [b.astype("float64") for b in mlp.intercepts_[1:]]

nodes, inits, prev = [], [], "input"
for i, (w, b) in enumerate(zip(weights, biases)):
    nodes.append(helper.make_node("MatMul", [prev, f"W{i}"], [f"m{i}"]))
    nodes.append(helper.make_node("Add", [f"m{i}", f"b{i}"], [f"z{i}"]))
    inits.append(helper.make_tensor(
        f"W{i}", TensorProto.FLOAT, list(w.shape), w.ravel().astype("float32").tolist()))
    inits.append(helper.make_tensor(
        f"b{i}", TensorProto.FLOAT, list(b.shape), b.ravel().astype("float32").tolist()))
    if i < len(weights) - 1:
        nodes.append(helper.make_node("Relu", [f"z{i}"], [f"a{i}"]))
        prev = f"a{i}"
    else:
        prev = f"z{i}"

# No Sigmoid on the last layer. MLPRegressor.predict is a linear read-out with
# no output activation, so adding one here makes the graph disagree with the
# pipeline it was folded from, which the parity check in the next cell catches.
# The loader clamps to 0-1 instead, which is where a risk score is defined.

graph = helper.make_graph(
    nodes=nodes,
    name="kryxai_risk",
    inputs=[helper.make_tensor_value_info(
        "input", TensorProto.FLOAT, [None, X.shape[1]])],
    # The output is the last layer's own tensor. Naming it "score" here while
    # nothing produces a tensor called "score" yields a file onnxruntime rejects.
    outputs=[helper.make_tensor_value_info(prev, TensorProto.FLOAT, [None, 1])],
    initializer=inits,
)

onnx_model = helper.make_model(
    graph,
    producer_name=f"kryxai-training-{NOTEBOOK_TAG}",
    opset_imports=[helper.make_opsetid("", 13)],
)
# StringStringEntryProto is built directly rather than through a helper:
# onnx.helper.make_string_string_entry does not exist in onnx 1.22, and a
# notebook that imports a helper the installed version lacks fails at the very
# last step, after the training is already done.
for key, value in [
    ("kryxai_feature_schema_version", str(fusion.FEATURE_SCHEMA_VERSION)),
    ("kryxai_feature_names", ",".join(names)),
    ("kryxai_labels", ",".join(D.BANDS)),
    ("kryxai_scaler_folded", "true"),
    ("kryxai_target_scale", str(TARGET_SCALE)),
]:
    onnx_model.metadata_props.append(onnx.StringStringEntryProto(key=key, value=value))

OUT = HERE / "dist"
OUT.mkdir(exist_ok=True)
onnx.save(onnx_model, str(OUT / "risk_model.onnx"))
print("wrote", OUT / "risk_model.onnx")

## 10. Verify the export three ways

An ONNX file that loads is not evidence that it computes the same thing.

1. It must agree with the fitted pipeline on held-out raw features.
2. It must agree when loaded through `LearnedRiskModel`, which is the path a
   scan actually takes.
3. Its metadata must match this build, or the installer will refuse it.

In [ ]:
import onnxruntime as ort

sess = ort.InferenceSession(str(OUT / "risk_model.onnx"), providers=["CPUExecutionProvider"])
meta = sess.get_modelmeta().custom_metadata_map
print("schema in file :", meta.get("kryxai_feature_schema_version"))
print("names in file  :", len(meta.get("kryxai_feature_names", "").split(",")))
assert int(meta["kryxai_feature_schema_version"]) == fusion.FEATURE_SCHEMA_VERSION
assert meta["kryxai_feature_names"].split(",") == names
print("metadata matches this build")

# 1. parity on RAW features, exactly as the runtime passes them
output_name = sess.get_outputs()[0].name
onnx_out = sess.run([output_name], {"input": X[test_idx].astype("float32")})[0].ravel()
sk_out = risk_model.predict(X[test_idx])
worst = float(np.max(np.abs(np.clip(onnx_out, 0.0, 1.0) - np.clip(sk_out, 0.0, 1.0))))
print(f"\noutput tensor    : {output_name}")
print(f"max |onnx - sklearn| on held-out rows: {worst:.3e}")
assert worst < 1e-4, "exported model disagrees with the fitted pipeline"
print("parity ok (compared after the loader's 0-1 clamp)")

# 2. the path a scan takes: LearnedRiskModel on a real features_for() vector,
#    built from each session's actual findings rather than an empty list.
#    Passing [] would exercise session_feature_row's stand-in row, which is a
#    different distribution from anything in the training set and would make the
#    scores below meaningless.
loader = fusion.LearnedRiskModel(str(OUT / "risk_model.onnx"))
print("\nloader status:", loader.status)
assert loader.status == "loaded", loader.status
assert loader.score(np.zeros(len(names), dtype="float32")) is not None, \
    "the runtime loader rejected its own model"

by_band = {}
for s in samples[:1] + [x for x in samples if x.band != "ok"][:1] + samples[-1:]:
    report = run_scan(s.path, Settings(blockchain_difficulty=1), persist=False).report
    for sess in report["sessions"]:
        fs = [f for f in report["findings"] if f.get("session_id") == sess["session_id"]]
        row = fusion.session_feature_row(sess, fs, {})
        by_band[s.band] = loader.score(row)
        print(f"  {s.band:12} -> {by_band[s.band]:.4f}")

# The model is a risk score, so a worse configuration must not score lower.
if "ok" in by_band and "weak_crypto" in by_band:
    assert by_band["ok"] <= by_band["weak_crypto"], \
        f"an acceptable session scored above a weak one: {by_band}"
if "exposure" in by_band:
    assert by_band["exposure"] >= by_band.get("weak_crypto", 0.0), \
        f"an exposed session scored below a merely weak one: {by_band}"
print("\nloader accepted raw features_for() vectors and ordered them correctly")

## 11. Model card

Written whether or not the model beat the baseline. The card records the
limitations as well as the metrics, because a card listing only advantages is
not a model card.

In [ ]:
import json
from pathlib import Path

card = {
    "name": "kryxai-risk-synthetic-v1",
    "feature_schema_version": fusion.FEATURE_SCHEMA_VERSION,
    "feature_names": names,
    "labels": list(D.BANDS),
    "target_scale": TARGET_SCALE,
    "trained_on": {
        "source": "training/dataset.py; synthetic captures generated in-notebook",
        "rows": int(X.shape[0]),
        "identities": int(len(set(groups))),
        "real_network_captures": 0,
        "notebook_tag": NOTEBOOK_TAG,
        "split": "GroupShuffleSplit over certificate identity, test_size=0.25",
    },
    "metrics": {
        "held_out_mae_bands": round(model_mae, 4),
        "held_out_accuracy": round(float(accuracy_score(truth, np.round(pred))), 4),
        "rule_baseline_mae_bands": round(base_mae, 4),
        "reaches_parity_with_rules": bool(REACHES_PARITY),
    },
    "limitations": [
        "Trained entirely on synthetic captures; never validated against real mail traffic.",
        "The labels are the injected weakness profiles, which are the same "
        "weaknesses the rule engine detects, so a good score here does not show "
        "the model adds anything the rules do not already have.",
        "Reaching parity on this corpus is evidence the pipeline is reproducible, "
        "not evidence of generalisation.",
        "TLS 1.3 certificate contents are encrypted, so those features are zero "
        "by construction and a TLS 1.3 session looks unassessed to the model.",
        "Key size, not key strength: a 2048-bit RSA key is scored as adequate, "
        "which is a policy statement, not a cryptographic measurement.",
        "Where the model and the rules disagree the report shows both scores "
        "rather than resolving the disagreement in favour of the model.",
    ],
}

card_path = HERE / "kryxai" / "scoring" / "model_card.json"
card_path.write_text(json.dumps(card, indent=2) + "\n", encoding="utf-8")
print(json.dumps(card["metrics"], indent=2))
print("\nwrote", card_path)

## 12. Download

`risk_model.onnx` is the trained model; `model_card.json` should travel with it.
On a local checkout, `install_model.py` does the validation and the copy:

```
python training/export/install_model.py dist/risk_model.onnx
```

In [ ]:
try:
    from google.colab import files
    files.download(str(OUT / "risk_model.onnx"))
    files.download(str(HERE / "kryxai" / "scoring" / "model_card.json"))
except ImportError:
    print("not running under Colab; artefacts are at:")
    for p in (OUT / "risk_model.onnx", HERE / "kryxai" / "scoring" / "model_card.json"):
        print("  ", p, p.stat().st_size, "bytes")

## What to report

State the held-out MAE, the rule baseline MAE, and whether the model reached
parity.

Be careful with the wording. A model that ties the rules on synthetic data has
shown that the training pipeline is reproducible. It has **not** shown that a
learned model is better than the rules, and this corpus cannot show that,
because the labels and the rules describe the same weaknesses. Say so rather
than presenting parity as an improvement.

If the model does not reach parity, leave it uninstalled. KryxAI's rule scoring
is the validated path.